<a href="https://colab.research.google.com/github/mercy7rotich-ops/CODY/blob/main/Week5_Model_Training_and_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Target, Features, and Problem Type Identification

* **Target (Output - $y$):**
  - `MedHouseVal` (Median house value for California districts, expressed in hundreds of thousands of dollars).
* **Inputs (Features - $X$):**
  - `MedInc`, `HouseAge`, `AveRooms`, `AveBedrms`, `Population`, `AveOccup`, `Latitude`, `Longitude`.

* **Is this a Regression or Classification problem?**
  - This is a **Regression** problem.
  - **Reason:** The target variable (`MedHouseVal`) is a continuous numerical value rather than a discrete category or class label. We are predicting a numeric price/value based on input features.

In [9]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# 1. Load dataset from OpenML
housing = fetch_openml(name="california_housing", version=1, as_frame=True)
X = housing.data
y = housing.target.astype(float)  # Ensures target values are numeric

# Keep only numeric columns and fill any missing values with the column median
X = X.select_dtypes(include=[np.number])
X = X.fillna(X.median())

# Drop any rows where the target value is missing
valid_idx = ~y.isna()
X = X[valid_idx]
y = y[valid_idx]

# 2. Split the dataset into training (80%) and test (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

# 3. Select and train two supervised learning models
# Model 1: Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Model 2: Random Forest Regressor
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# 4. Use both models to make predictions on the test data
lr_preds = lr_model.predict(X_test)
rf_preds = rf_model.predict(X_test)

print("\nModels trained and predictions generated successfully!")

Training set shape: (16512, 8)
Test set shape: (4128, 8)

Models trained and predictions generated successfully!


In [10]:
# Evaluate Linear Regression
lr_mse = mean_squared_error(y_test, lr_preds)
lr_rmse = np.sqrt(lr_mse)
lr_r2 = r2_score(y_test, lr_preds)

# Evaluate Random Forest Regressor
rf_mse = mean_squared_error(y_test, rf_preds)
rf_rmse = np.sqrt(rf_mse)
rf_r2 = r2_score(y_test, rf_preds)

# Display performance comparison table
evaluation_results = pd.DataFrame({
    'Metric': ['Mean Squared Error (MSE)', 'Root Mean Squared Error (RMSE)', 'R-squared (R2 Score)'],
    'Linear Regression': [lr_mse, lr_rmse, lr_r2],
    'Random Forest Regressor': [rf_mse, rf_rmse, rf_r2]
})

print("=== Model Performance Comparison ===")
display(evaluation_results)

=== Model Performance Comparison ===


,Metric,Linear Regression,Random Forest Regressor
0,Mean Squared Error (MSE),5.059928e+09,2.488060e+09
1,Root Mean Squared Error (RMSE),7.113317e+04,4.988046e+04
2,R-squared (R2 Score),6.138665e-01,8.101310e-01


## 2. Discussion of Results

* **Performance Comparison:**
  - The **Random Forest Regressor** significantly outperforms the Linear Regression model across all metrics. It yields a much lower RMSE and a higher $R^2$ score.
  - The higher $R^2$ score for Random Forest indicates that it captures a larger portion of the variance in housing prices, while Linear Regression's performance is limited by its assumption of a strictly linear relationship between features and the target.

* **Conclusion:**
  - Complex real estate relationships often involve non-linear interactions between variables (e.g., location, income, and room counts), which tree-based ensemble methods like Random Forest handle much more effectively than basic linear models.